# B05 — QParser Quality Eval (via `/qparser` API)

Sends each question + its premises to `/qparser` and inspects the returned `QuerySpec`.

- **Input:** `Logic_Based_Educational_Queries.json` — questions carry embedded `A.`/`A)` options.
- **Pipeline:** premises → schema → question-side parser → `QuerySpec` (no solving).
- **Scoring:** format-detection accuracy, solver_mode distribution, can_interpretation split,
  option_type breakdown, % supported, latency.


In [ ]:
import json, re, time, asyncio, statistics
from pathlib import Path
from collections import Counter

import httpx

# --- endpoint -------------------------------------------------------------
API_BASE    = "https://api.iamphuckhang.dev"
QPARSER_URL = f"{API_BASE}/qparser"

# --- run size -------------------------------------------------------------
N_SAMPLES   = 10       # how many questions to eval (None = all 808)
CONCURRENCY = 8
TIMEOUT     = 120.0

# --- locate dataset -------------------------------------------------------
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src/exact/datasets/exact").exists():
    ROOT = ROOT.parent
DATA = ROOT / "src/exact/datasets/exact"
assert DATA.exists(), f"dataset dir not found from {Path.cwd()}"
print("dataset :", DATA)
print("endpoint:", QPARSER_URL)


In [ ]:
raw = json.load(open(DATA / "Logic_Based_Educational_Queries.json"))

_OPTION_LINE = re.compile(r"^\s*([A-E])[.)]\s*(.*)$")

def has_embedded_options(question_text: str) -> bool:
    """Surface check: does this question carry A./A) option lines?"""
    return any(_OPTION_LINE.match(line) for line in question_text.split("\n"))

def flatten_instances(groups: list) -> list[dict]:
    instances = []
    for g_idx, group in enumerate(groups):
        for q_idx, question in enumerate(group["questions"]):
            instances.append({
                "id": f"logic_{g_idx:04d}_{q_idx:02d}",
                "premises": group["premises-NL"],
                "question": question,
                # Surface gold format: mcq if options are embedded, else polar.
                "surface_format": "mcq" if has_embedded_options(question) else "polar",
            })
    return instances

instances = flatten_instances(raw)
surf = Counter(i["surface_format"] for i in instances)
print(f"{len(instances)} questions  (surface mcq: {surf['mcq']}, polar: {surf['polar']})")
print("\nExample:", instances[0]["id"])
print(instances[0]["question"][:300])


In [ ]:
async def call(client, sem, inst):
    async with sem:
        t0 = time.perf_counter()
        err, body = None, None
        try:
            r = await client.post(
                QPARSER_URL,
                json={"question": inst["question"], "premises": inst["premises"]},
                timeout=TIMEOUT,
            )
            r.raise_for_status()
            body = r.json()
        except Exception as e:
            err = repr(e)
        dt = time.perf_counter() - t0

    return {
        "id": inst["id"],
        "surface_format": inst["surface_format"],
        "spec": body,
        "latency": dt,
        "error": err,
    }


async def run_eval(instances):
    sem = asyncio.Semaphore(CONCURRENCY)
    done = 0
    async with httpx.AsyncClient() as client:
        async def wrapped(inst):
            nonlocal done
            res = await call(client, sem, inst)
            done += 1
            if done % 5 == 0 or done == len(instances):
                print(f"  {done}/{len(instances)}", end="\r")
            return res
        return await asyncio.gather(*(wrapped(i) for i in instances))


In [ ]:
subset = instances[:N_SAMPLES] if N_SAMPLES else instances
print(f"Evaluating {len(subset)} questions at concurrency {CONCURRENCY}...")
t0 = time.perf_counter()
results = await run_eval(subset)
wall = time.perf_counter() - t0

errors  = [r for r in results if r["error"]]
success = [r for r in results if not r["error"]]
print(f"\nSuccess : {len(success)}/{len(results)}")
print(f"Errors  : {len(errors)}")
print(f"Wall    : {wall:.1f}s")


In [ ]:
if errors:
    print("=== Errors ===")
    for r in errors:
        print(f"  [{r['id']}] {r['error']}")
    print()

# --- Per-question QuerySpec ---
print("=== QuerySpec per question ===")
for r in success:
    s = r["spec"]
    flag = "✓" if s["supported"] else "✗"
    print(f"\n{flag} {r['id']}  surface={r['surface_format']}  ->  "
          f"{s['question_format']}/{s['solver_mode']}  can={s['can_interpretation']}  ({r['latency']:.1f}s)")
    if s["main_claim_text"]:
        print(f"    claim: {s['main_claim_text']}")
        print(f"    fol  : {s['main_claim_fol']}")
    for c in s["option_claims"]:
        detail = c["claim_text"] or c["raw_fol"] or (f"premises {c['premise_indices']}" if c['premise_indices'] else c['ynu_value'])
        print(f"    {c['label']}. [{c['option_type']}] {detail}")
        if c["fol"]:
            print(f"        fol: {c['fol']}")
    if s["issues"]:
        for issue in s["issues"]:
            print(f"    !! {issue}")


In [ ]:
n = len(success)
if n == 0:
    print("No successful results.")
else:
    # --- Format-detection accuracy (predicted format vs surface gold) ---
    #  surface 'mcq' should map to predicted 'mcq'; surface 'polar' to 'polar' or 'open_wh'.
    correct_fmt = 0
    for r in success:
        pred = r["spec"]["question_format"]
        if r["surface_format"] == "mcq":
            correct_fmt += pred == "mcq"
        else:
            correct_fmt += pred in ("polar", "open_wh")
    print("=== Format detection ===")
    print(f"  Accuracy : {correct_fmt/n:.1%}  ({correct_fmt}/{n})")

    # --- Distributions ---
    fmt_dist  = Counter(r["spec"]["question_format"] for r in success)
    mode_dist = Counter(r["spec"]["solver_mode"] for r in success)
    can_dist  = Counter(r["spec"]["can_interpretation"] for r in success)
    supported = sum(r["spec"]["supported"] for r in success)

    print("\n=== question_format ===")
    for k, v in fmt_dist.most_common(): print(f"  {k:12s}: {v}")
    print("\n=== solver_mode ===")
    for k, v in mode_dist.most_common(): print(f"  {k:20s}: {v}")
    print("\n=== can_interpretation ===")
    for k, v in can_dist.most_common(): print(f"  {k:14s}: {v}")
    print(f"\n=== Supported ===\n  {supported}/{n}  ({supported/n:.1%})")

    # --- option_type breakdown ---
    opt_types = Counter()
    for r in success:
        for c in r["spec"]["option_claims"]:
            opt_types[c["option_type"]] += 1
    if opt_types:
        print("\n=== option_type breakdown ===")
        for k, v in opt_types.most_common(): print(f"  {k:18s}: {v}")

    # --- issue tags ---
    issue_tags = Counter()
    for r in success:
        for issue in r["spec"]["issues"]:
            issue_tags[issue.split(":")[0]] += 1
    if issue_tags:
        print("\n=== issue tags ===")
        for k, v in issue_tags.most_common(): print(f"  {k:30s}: {v}")

    lat = [r["latency"] for r in success]
    print(f"\n=== Latency ===")
    print(f"  mean={statistics.mean(lat):.2f}s  p50={statistics.median(lat):.2f}s  max={max(lat):.2f}s")


## Notes
- Set `N_SAMPLES = None` to run all 808 questions.
- `/qparser` only classifies + compiles claims — it does **not** solve. Use `/predict` or `/z3` for answers.
- `can_interpretation`: `meta_inference` = "which can be inferred"; `object_modal` = "Can X do Y?".
- Deferred modes (`strongest_conclusion` / `fewest_premise` / `premise_selection`) show as
  `supported=False` with a `QUERY_MODE_DEFERRED` issue — they are classified but not solved in v1.
- To inspect one result: `next(r for r in results if r['id'] == 'logic_0000_00')['spec']`.
